In [2]:
import boto3

In [3]:
ec2_resource = boto3.resource('ec2', region_name='us-east-1')

# create VPC
vpc = ec2_resource.create_vpc(CidrBlock='192.168.0.0/16')
vpc.wait_until_available()
vpc.create_tags(
    Tags=[
        {
            "Key": "Name",
            "Value": "my-vpc"
        }
    ]
)
vpcName = vpc.tags[0]['Value']
print(vpcName + ' ' + vpc.id)


ClientError: An error occurred (RequestExpired) when calling the CreateVpc operation: Request has expired.

In [ ]:
# create internet gateway and attach it to the VPC
internet_gateway = ec2_resource.create_internet_gateway()
vpc.attach_internet_gateway(InternetGatewayId=internet_gateway.id)
print(internet_gateway.id)

# create a route table and a public route
route_table = vpc.create_route_table()
route = route_table.create_route(
    DestinationCidrBlock='0.0.0.0/0',
    GatewayId=internet_gateway.id
)
print(route_table.id)

In [ ]:
# create a public subnet
subnet = ec2_resource.create_subnet(
    CidrBlock='192.168.1.16/28',
    AvailabilityZone='us-east-1a',
    VpcId=vpc.id
)
subnet.create_tags(
    Tags=[
        {
            "Key": "Name",
            "Value": "my-public-subnet-1a"
        }
    ]
)
subnetName = subnet.tags[0]['Value']
print(subnetName + ' ' + subnet.id + ' ' + subnet.availability_zone)
route_table.associate_with_subnet(SubnetId=subnet.id)


In [ ]:
# create a private subnet
subnet = ec2_resource.create_subnet(
    CidrBlock='',
    AvailabilityZone='us-east-1a',
    VpcId=vpc.id
)
subnet.create_tags(
    Tags=[
        {
            "Key": "Name",
            "Value": "my-private-subnet-1a"
        }
    ]
)

In [ ]:
# find the subnets associated with the route table
route_table = ec2_resource.RouteTable(route_table.id)
associations = route_table.associations_attribute
for i in range(len(associations)):
    if i == 0:
        subnets = associations[0]['SubnetId']
    else:
        subnets = subnets + ', ' + associations[i]['SubnetId']

print(subnets)

In [ ]:
# create two security groups for these two subnets.
private_security_group = ec2_resource.create_security_group(
    GroupName='my-vpc-private-subnet-security-group', 
    Description='my vpc private subnet security group', 
    VpcId=vpc.id)

private_security_group.authorize_ingress(
    CidrIp=vpc.cidr_block,   # allowing traffic from internal of vpc     
    IpProtocol='-1'  # allowing all protocol and all ports
)

print(private_security_group.id)

public_security_group = ec2_resource.create_security_group(
    GroupName='my-vpc-public-subnet-security-group', 
    Description='my vpc public subnet security group', 
    VpcId=vpc.id)

public_security_group.authorize_ingress(
    CidrIp='',   # allowing traffic from anywhere
    IpProtocol='tcp',
    FromPort=80,
    ToPort=80
)


In [ ]:
# Launch two instances into these two subnets.
instances = ec2_resource.create_instances(
     ImageId='ami-038f1ca1bd58a5790',
     MinCount=1,
     MaxCount=1,
     InstanceType='t2.micro',
     KeyName='ec2keypair',
     NetworkInterfaces=[{
        'SubnetId': subnet.id,
        'DeviceIndex': 0,
        'AssociatePublicIpAddress': True,
        'Groups': [http_security_group.id]
    }],
)